In [2]:
%load_ext autoreload
%autoreload 2

import mujoco
from swarmbots.swarm_bots_env import SwarmBotsEnv
from swarmbots.swarm.simple_swarm import SimpleSwarm
from swarmbots.scenarios.obstacle_street_scenario import ObstacleStreetScenario
from rendering import display_video

import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

swarm = SimpleSwarm(connection_torquescale=10)
scenario = ObstacleStreetScenario(swarm, payload_type=None, seed=42)

opt = mujoco.MjvOption()

env = SwarmBotsEnv(
    scenario=scenario,
    render_mode="rgb_array",
    width=640,
    height=480,
    camera=0,
    scene_option=opt,
    action_repeat=15,
)

rng = np.random.default_rng()


frames = []
for i in range(1):
    done = False
    obs, info = env.reset()

    while not done:
        # Random action
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)

        frame = env.render()
        if frame is not None:
            frames.append(frame)

        done = terminated or truncated

        if info:
            print('err ' + str(env.data.time))


        # if len(frames) % 50 == 0:
        #     env.data.eq_active[:] = 0
        #     env.data.eq_active[rng.integers(low=0, high=len(env.data.eq_active))] = 1

    print(f"Recorded {len(frames)} frames")
    for _ in range(15):
        frames.append(np.zeros_like(frames[0]))

# env.close()
display_video(frames, 30)

In [15]:
is_active, _, _ = env.swarm_connections.get_active_connections()
is_active

array([[False, False,  True, False, False, False],
       [False, False, False,  True, False, False],
       [False, False, False, False, False, False],
       [False, False, False, False, False, False],
       [False, False, False, False, False, False]])

In [26]:
action = np.asarray(env.action_space.sample()['connectors'], dtype=bool)
action

array([[ True, False,  True,  True,  True, False],
       [False, False, False, False,  True, False],
       [False,  True, False, False,  True,  True],
       [ True, False, False, False, False,  True],
       [False, False, False,  True, False,  True]])

array([[0, 1, 0, 0, 0, 1],
       [1, 1, 1, 1, 0, 1],
       [1, 0, 1, 1, 0, 0],
       [0, 1, 1, 1, 1, 0],
       [1, 1, 1, 0, 1, 0]])

In [34]:
np.stack(np.where(np.logical_and(action, np.logical_not(is_active)))).T

array([[0, 0],
       [0, 3],
       [0, 4],
       [1, 4],
       [2, 1],
       [2, 4],
       [2, 5],
       [3, 0],
       [3, 5],
       [4, 3],
       [4, 5]])

In [31]:
np.where(np.logical_and(is_active, np.logical_not(action)))

(array([1]), array([3]))

In [19]:
type(env.data.warning[mujoco.mjtWarning.mjWARN_BADQACC].lastinfo)

int

In [5]:
env.model.eq_data[:5]

array([[0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.]])

In [3]:
env.scenario.data.qpos.shape

(38,)